# Notebook 02: Current Sheet Alignment and Threshold Crossing — Full Refinement

This notebook upgrades the current-sheet experiment into a cleaner spatial analysis.

It now includes:

- a Harris-sheet-inspired magnetic field
- a localized transverse perturbation
- cross-sheet cosine maps
- 45° contour boundaries drawn directly on cosine maps
- threshold masks
- thickness sweeps
- center-vs-domain summary curves
- a simple geometric width estimate for the 45° boundary

Core geometric gate:

\[
\cos\theta \geq \frac{1}{\sqrt{1^2 + 1^2}}
\]

Interpretation used here:

- high cosine \(\to\) aligned field
- low cosine \(\to\) strong cross-sheet shear
- the 45° contour marks the spatial boundary where the field transitions across the gate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

THRESHOLD = 1 / np.sqrt(2)

nx, ny = 300, 300
x = np.linspace(-5.0, 5.0, nx)
y = np.linspace(-3.0, 3.0, ny)
X, Y = np.meshgrid(x, y)

print("45° threshold =", THRESHOLD)
print("grid shape =", X.shape)

## 1. Field model and helpers

In [ ]:
def magnetic_field(X, Y, delta=0.5, eps=0.25, k=1.2, sigma=1.0):
    Bx = np.tanh(Y / delta)
    By = eps * np.sin(k * X) * np.exp(-(Y**2) / (sigma**2))
    return Bx, By

def normalize_field(Bx, By):
    mag = np.sqrt(Bx**2 + By**2)
    mag = np.where(mag == 0, 1.0, mag)
    return Bx / mag, By / mag

def cross_sheet_cosine(Bx_hat, By_hat, shift=3):
    Bx_up = np.roll(Bx_hat, -shift, axis=0)
    By_up = np.roll(By_hat, -shift, axis=0)

    Bx_dn = np.roll(Bx_hat, shift, axis=0)
    By_dn = np.roll(By_hat, shift, axis=0)

    cos_map = Bx_up * Bx_dn + By_up * By_dn

    # Exclude wrapped rows introduced by np.roll
    cos_map[:shift, :] = np.nan
    cos_map[-shift:, :] = np.nan
    return cos_map

def threshold_mask(cos_map, threshold=THRESHOLD):
    return cos_map >= threshold

def finite_mean(arr):
    return np.nanmean(arr)

def estimate_boundary_width_y(cos_map, x, y, threshold=THRESHOLD, x_window=(-1.0, 1.0)):
    # Estimate the y-width of the region where cos < threshold in a central x-window.
    x_sel = (x >= x_window[0]) & (x <= x_window[1])
    sub = cos_map[:, x_sel]
    bad = sub < threshold
    frac_per_y = np.nanmean(bad, axis=1)

    y_bad = y[frac_per_y > 0]
    if len(y_bad) == 0:
        return 0.0
    return float(y_bad.max() - y_bad.min())

## 2. Example field and one-sheet geometry

This first section shows the spatial field before the sweep.

In [ ]:
delta_example = 0.6
Bx, By = magnetic_field(X, Y, delta=delta_example, eps=0.25, k=1.2, sigma=1.0)
Bx_hat, By_hat = normalize_field(Bx, By)

skip = 10
plt.figure(figsize=(8, 5))
plt.quiver(X[::skip, ::skip], Y[::skip, ::skip], Bx_hat[::skip, ::skip], By_hat[::skip, ::skip])
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Normalized Magnetic Field (delta={delta_example})")
plt.show()

In [ ]:
mid_x = nx // 2

plt.figure(figsize=(8, 5))
plt.plot(y, Bx[:, mid_x], label="Bx at x=0")
plt.xlabel("y")
plt.ylabel("Bx")
plt.title("Field Reversal Across the Current Sheet")
plt.legend()
plt.show()

## 3. Cross-sheet cosine map with 45° contour

For each point we compare the normalized field above and below the point:

\[
\cos\theta(x,y)=\hat B(x,y+\Delta y)\cdot \hat B(x,y-\Delta y).
\]

The white dashed contour is the explicit 45° boundary.

In [ ]:
cos_map_example = cross_sheet_cosine(Bx_hat, By_hat, shift=3)

plt.figure(figsize=(8, 5))
plt.imshow(
    cos_map_example,
    extent=[x.min(), x.max(), y.min(), y.max()],
    origin="lower",
    aspect="auto"
)
plt.colorbar(label="cross-sheet cosine")
plt.contour(
    X, Y, cos_map_example,
    levels=[THRESHOLD],
    colors="white",
    linewidths=1.8,
    linestyles="--"
)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Cross-Sheet Cosine Map (delta={delta_example})")
plt.show()

In [ ]:
mask_example = threshold_mask(cos_map_example)

plt.figure(figsize=(8, 5))
plt.imshow(
    mask_example,
    extent=[x.min(), x.max(), y.min(), y.max()],
    origin="lower",
    aspect="auto"
)
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"45° Threshold Mask (delta={delta_example})")
plt.show()

## 4. Thickness sweep

We vary the current-sheet thickness \(\delta\) and record:

- domain-wide threshold fraction
- center-region threshold fraction
- localization ratio = center fraction / total fraction
- estimated 45° boundary width in \(y\)

As \(\delta\) decreases, the reversal sharpens.

In [ ]:
deltas = [1.2, 0.8, 0.6, 0.4, 0.25]
shift = 3
center_width = 0.45

results = []

for delta in deltas:
    Bx, By = magnetic_field(X, Y, delta=delta, eps=0.25, k=1.2, sigma=1.0)
    Bx_hat, By_hat = normalize_field(Bx, By)

    cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=shift)
    mask = threshold_mask(cos_map)

    total_fraction = float(np.nanmean(mask))

    center_region = np.abs(Y) < center_width
    center_fraction = float(np.nanmean(mask[center_region]))

    localization_ratio = center_fraction / total_fraction if total_fraction > 0 else np.nan
    boundary_width = estimate_boundary_width_y(cos_map, x, y, threshold=THRESHOLD, x_window=(-1.0, 1.0))

    results.append({
        "delta": float(delta),
        "cos_map": cos_map,
        "mask": mask,
        "total_fraction": total_fraction,
        "center_fraction": center_fraction,
        "localization_ratio": float(localization_ratio),
        "boundary_width_y": float(boundary_width),
    })

summary = [
    {
        "delta": r["delta"],
        "total_fraction": r["total_fraction"],
        "center_fraction": r["center_fraction"],
        "localization_ratio": r["localization_ratio"],
        "boundary_width_y": r["boundary_width_y"],
    }
    for r in results
]

summary

## 5. Cosine maps for all thickness values

Each panel includes the 45° contour. This is the main geometric refinement over the earlier version.

In [ ]:
for r in results:
    plt.figure(figsize=(8, 4.8))
    plt.imshow(
        r["cos_map"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.colorbar(label="cross-sheet cosine")
    plt.contour(
        X, Y, r["cos_map"],
        levels=[THRESHOLD],
        colors="white",
        linewidths=1.8,
        linestyles="--"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Cross-Sheet Cosine Map (delta={r['delta']})")
    plt.show()

## 6. Threshold masks for all thickness values

The mask shows where the field passes the 45° gate. The failed band isolates the strongest shear region.

In [ ]:
for r in results:
    plt.figure(figsize=(8, 4.8))
    plt.imshow(
        r["mask"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"45° Threshold Mask (delta={r['delta']})")
    plt.show()

## 7. Summary curves

These are the compact quantitative outputs for the notebook.

In [ ]:
delta_vals = np.array([r["delta"] for r in results])
total_vals = np.array([r["total_fraction"] for r in results])
center_vals = np.array([r["center_fraction"] for r in results])
local_vals = np.array([r["localization_ratio"] for r in results])
width_vals = np.array([r["boundary_width_y"] for r in results])

plt.figure(figsize=(8, 5))
plt.plot(delta_vals, total_vals, marker="o", label="domain-wide threshold fraction")
plt.plot(delta_vals, center_vals, marker="s", label="center-region threshold fraction")
plt.xlabel("sheet thickness delta")
plt.ylabel("fraction passing 45°")
plt.title("Threshold Fraction vs Sheet Thickness")
plt.legend()
plt.gca().invert_xaxis()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(delta_vals, local_vals, marker="o")
plt.xlabel("sheet thickness delta")
plt.ylabel("localization ratio")
plt.title("Localization Ratio vs Sheet Thickness")
plt.gca().invert_xaxis()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(delta_vals, width_vals, marker="o")
plt.xlabel("sheet thickness delta")
plt.ylabel("estimated 45° boundary width in y")
plt.title("45° Boundary Width vs Sheet Thickness")
plt.gca().invert_xaxis()
plt.show()

## 8. Compact printed summary

In [ ]:
for r in summary:
    print(
        f"delta={r['delta']:.2f} | "
        f"total_fraction={r['total_fraction']:.4f} | "
        f"center_fraction={r['center_fraction']:.4f} | "
        f"localization_ratio={r['localization_ratio']:.4f} | "
        f"boundary_width_y={r['boundary_width_y']:.4f}"
    )

## 9. Interpretation

This refined notebook supports a cleaner statement than the earlier version:

> The 45° gate does not create reconnection by itself.  
> It acts as a geometric separator that isolates the strong-shear current-sheet region from the broadly aligned background field.

The main upgrade is the explicit contour:

- the cosine map gives the full scalar field
- the white dashed contour gives the exact 45° boundary
- the width estimate turns that boundary into a tracked quantity

This is best interpreted as a **geometric precursor analysis**, not yet a full dynamical reconnection model.

A natural next notebook is to seed instability along the sheet so that the single failed band can fragment into multiple structures.